# Chicago Crime Data - Cleaning

Standard machine learning data cleaning steps, plus feature engineering (year, month, day of week, hour, etc.) for downstream modeling and EDA.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
df = pd.read_csv('../../data/raw/crimes.csv')
print("Initial shape:", df.shape)
df.head(3)

Initial shape: (2477115, 22)


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,14135339,JK179351,03/13/2026 12:00:00 AM,075XX S KINGSTON AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,False,False,421,4.0,7.0,43.0,06,1194503.0,1855492.0,2026,03/20/2026 03:43:34 PM,41.758396,-87.562723,"(41.758395718, -87.562722643)"
1,14135179,JK179134,03/13/2026 12:00:00 AM,050XX N MARINE DR,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE - GARAGE,False,False,2024,20.0,48.0,3.0,14,1169716.0,1933892.0,2026,03/20/2026 03:43:34 PM,41.974105,-87.651281,"(41.974104832, -87.651281486)"
2,14138214,JK183002,03/13/2026 12:00:00 AM,075XX S STONY ISLAND AVE,0281,CRIMINAL SEXUAL ASSAULT,NON-AGGRAVATED,HOSPITAL BUILDING / GROUNDS,False,False,411,4.0,8.0,43.0,02,1188234.0,1855158.0,2026,03/20/2026 03:43:34 PM,41.757631,-87.585708,"(41.757630995, -87.585708249)"


## 1. Drop Redundant Columns

Based on data understanding: Case Number, Location, X Coordinate, Y Coordinate, Updated On.

In [2]:
cols_to_drop = ['Case Number', 'Location', 'X Coordinate', 'Y Coordinate', 'Updated On']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print("After dropping redundant columns:", df.shape)

After dropping redundant columns: (2477115, 17)


## 2. Handle Missing Values

- Latitude/Longitude: Drop rows with missing coordinates (needed for spatial analysis)
- Location Description: Fill with "UNKNOWN" (categorical)
- Community Area: Fill with -1 for unknown

In [3]:
print("Missing before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Drop rows with missing Lat/Lon (needed for mapping and location-based predictions)
n_before = len(df)
df = df.dropna(subset=['Latitude', 'Longitude'])
print(f"\nDropped {n_before - len(df)} rows with missing coordinates")

# Fill Location Description with UNKNOWN
df['Location Description'] = df['Location Description'].fillna('UNKNOWN')

# Fill Community Area with -1 (unknown)
df['Community Area'] = df['Community Area'].fillna(-1).astype(int)
print("\nMissing after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing before cleaning:
Location Description    13003
District                    1
Ward                       54
Community Area            203
Latitude                35605
Longitude               35605
dtype: int64

Dropped 35605 rows with missing coordinates

Missing after cleaning:
District     1
Ward        52
dtype: int64


## 3. Parse Date and Add Temporal Features

Extract year, month, day of week, hour, quarter, weekend flag for timing analysis and modeling.

In [4]:
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
df = df.dropna(subset=['Date'])  # drop any unparseable dates

# Add temporal features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
df['DayOfWeekName'] = df['Date'].dt.day_name()
df['Hour'] = df['Date'].dt.hour
df['Quarter'] = df['Date'].dt.quarter
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6])

print("Sample with new columns:")
df[['Date', 'Year', 'Month', 'DayOfWeekName', 'Hour', 'Quarter', 'IsWeekend']].head()

Sample with new columns:


,Date,Year,Month,DayOfWeekName,Hour,Quarter,IsWeekend
0,2026-03-13,2026,3,Friday,0,1,False
1,2026-03-13,2026,3,Friday,0,1,False
2,2026-03-13,2026,3,Friday,0,1,False
3,2026-03-13,2026,3,Friday,0,1,False
4,2026-03-13,2026,3,Friday,0,1,False


## 3b. Add Spatial Bins and Crime Category (for Modeling)

- **Lat_bin, Lon_bin**: 0.02-degree grid (~2km) for spatial features to predict Beat
- **HourBin**: 6 bins (0-4, 4-8, 8-12, 12-16, 16-20, 20-24) for time-of-day prediction
- **Crime_Category**: Group Primary Type into 5 categories for better prediction accuracy

In [5]:
# Spatial bins (0.02 deg ~ 2km for Chicago)
df['Lat_bin'] = (df['Latitude'] // 0.02).astype(int)
df['Lon_bin'] = (df['Longitude'] // 0.02).astype(int)

# HourBin: 6 bins for time-of-day prediction
def get_hour_bin(h):
    if 0 <= h < 4: return 0
    if 4 <= h < 8: return 1
    if 8 <= h < 12: return 2
    if 12 <= h < 16: return 3
    if 16 <= h < 20: return 4
    return 5
df['HourBin'] = df['Hour'].apply(get_hour_bin)

# Crime_Category: 5 groups for modeling (aligns with prediction task)
CRIME_CATEGORY_MAP = {
    'Violent': ['BATTERY', 'ASSAULT', 'ROBBERY', 'CRIMINAL SEXUAL ASSAULT', 'HOMICIDE', 'KIDNAPPING'],
    'Theft': ['THEFT', 'MOTOR VEHICLE THEFT', 'BURGLARY'],
    'Property': ['CRIMINAL DAMAGE', 'CRIMINAL TRESPASS', 'ARSON'],
    'Narcotics': ['NARCOTICS', 'OTHER NARCOTIC VIOLATION'],
    'Other': []  # default for all others
}
def map_crime_category(pt):
    for cat, types in CRIME_CATEGORY_MAP.items():
        if pt in types: return cat
    return 'Other'
df['Crime_Category'] = df['Primary Type'].apply(map_crime_category)

print("Crime_Category distribution:", df['Crime_Category'].value_counts().to_dict())
print("Lat_bin range:", df['Lat_bin'].min(), "-", df['Lat_bin'].max())
print("Lon_bin range:", df['Lon_bin'].min(), "-", df['Lon_bin'].max())

Crime_Category distribution: {'Theft': 801056, 'Violent': 765149, 'Other': 459614, 'Property': 331226, 'Narcotics': 84465}
Lat_bin range: 1830 - 2101
Lon_bin range: -4585 - -4377


## 4. Handle Duplicates

Remove duplicate records by ID.

In [6]:
dup_count = df.duplicated(subset=['ID']).sum()
df = df.drop_duplicates(subset=['ID'])
print(f"Removed {dup_count} duplicate rows by ID. Shape: {df.shape}")

Removed 0 duplicate rows by ID. Shape: (2441510, 28)


## 5. Ensure Correct Data Types

Convert Arrest/Domestic to int for ML; ensure Ward and Community Area are int.

In [7]:
df['Arrest'] = df['Arrest'].astype(int)
df['Domestic'] = df['Domestic'].astype(int)
df['Ward'] = df['Ward'].fillna(-1).astype(int)
print("Data types check:")
df.dtypes

Data types check:


ID                               int64
Date                    datetime64[us]
Block                              str
IUCR                               str
Primary Type                       str
Description                        str
Location Description               str
Arrest                           int64
Domestic                         int64
Beat                             int64
District                       float64
Ward                             int64
Community Area                   int64
FBI Code                           str
Year                             int32
Latitude                       float64
Longitude                      float64
Month                            int32
DayOfWeek                        int32
DayOfWeekName                      str
Hour                             int32
Quarter                          int32
WeekOfYear                       int64
IsWeekend                         bool
Lat_bin                          int64
Lon_bin                  

## 6. Handle Outliers (Coordinates)

Filter out points outside Chicago bounds. Chicago roughly: Lat 41.6-42.0, Lon -87.95 to -87.5.

In [8]:
CHICAGO_LAT = (41.6, 42.1)
CHICAGO_LON = (-87.95, -87.5)
mask = (df['Latitude'].between(*CHICAGO_LAT)) & (df['Longitude'].between(*CHICAGO_LON))
outliers = (~mask).sum()
df = df[mask]
print(f"Removed {outliers} rows with coordinates outside Chicago. Shape: {df.shape}")

Removed 4 rows with coordinates outside Chicago. Shape: (2441506, 28)


## 7. Save Cleaned Data

Store the cleaned dataset for EDA and modeling.

In [9]:
Path('../../data/processed').mkdir(parents=True, exist_ok=True)
df.to_csv('../../data/processed/crimes_cleaned.csv', index=False)
print("Saved to ../../data/processed/crimes_cleaned.csv")
print("Final shape:", df.shape)
df.head()

Saved to ../../data/processed/crimes_cleaned.csv
Final shape: (2441506, 28)


,ID,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,Year,Latitude,Longitude,Month,DayOfWeek,DayOfWeekName,Hour,Quarter,WeekOfYear,IsWeekend,Lat_bin,Lon_bin,HourBin,Crime_Category
0,14135339,2026-03-13,075XX S KINGSTON AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,0,0,421,4.0,7,43,06,2026,41.758396,-87.562723,3,4,Friday,0,1,11,False,2087,-4379,0,Theft
1,14135179,2026-03-13,050XX N MARINE DR,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE - GARAGE,0,0,2024,20.0,48,3,14,2026,41.974105,-87.651281,3,4,Friday,0,1,11,False,2098,-4383,0,Property
2,14138214,2026-03-13,075XX S STONY ISLAND AVE,0281,CRIMINAL SEXUAL ASSAULT,NON-AGGRAVATED,HOSPITAL BUILDING / GROUNDS,0,0,411,4.0,8,43,02,2026,41.757631,-87.585708,3,4,Friday,0,1,11,False,2087,-4380,0,Violent
3,14136294,2026-03-13,002XX E HURON ST,0281,CRIMINAL SEXUAL ASSAULT,NON-AGGRAVATED,HOSPITAL BUILDING / GROUNDS,0,0,1834,18.0,2,8,02,2026,41.895003,-87.621528,3,4,Friday,0,1,11,False,2094,-4382,0,Violent
4,14138215,2026-03-13,066XX W BELDEN AVE,1242,DECEPTIVE PRACTICE,COMPUTER FRAUD,OTHER (SPECIFY),0,0,2512,25.0,29,18,11,2026,41.921200,-87.792007,3,4,Friday,0,1,11,False,2096,-4390,0,Other
